# InfiniSplat v3 (SplatAnything) on Colab: any views, any scene → 3D Gaussians

<a href="https://colab.research.google.com/github/Cortex78/InfiniSplat/blob/main/notebooks/InfiniSplat_v3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook installs and runs [InfiniSplat v3 (SplatAnything)](https://github.com/Cortex78/InfiniSplat/tree/main/InfiniSplat_v3) ([project page](https://zju3dv.github.io/SplatAnything/), [paper](https://arxiv.org/html/2611.08264), [weights](https://huggingface.co/ritianyu/SplatAnything)) on an **A100**. v3 turns overlapping photos of objects, people, rooms, streets or aerial views into one 3D Gaussian scene, with or without camera poses. It also handles multi-camera rigs and 360° panoramas.

**How it works:**
* A WorldMirror-2 geometry model predicts depth (and the cameras, when you have none).
* A Gaussian model decodes per-pixel Gaussians.
* *Gaussian spacing normalization* puts every scene on a common scale, so one model handles every kind of scene.
* The per-pixel Gaussians are merged in voxels into one scene.

It is feed-forward: there is no per-scene optimisation.

| Cells | What they do |
| --- | --- |
| **1 · Setup** | Check the GPU, get the code, install the environment, download the weights (16.4 GB) |
| **2 · Settings** | Every tunable parameter of the model, the filtering, the export, memory use and the viewer |
| **3 · Reconstruct** | The six bundled scenes, your photos, a video, a 360° panorama, or the last input again with new settings |
| **4 · Results** | Download, or copy everything to Google Drive |

**Before you start:** choose *Runtime → Change runtime type → A100 GPU*.

**Everything runs on the GPU:**
* The install cell checks that PyTorch can use CUDA, and prints the measured bf16 throughput and whether FlashAttention is available.
* Inference is pinned to the GPU. If there is no usable GPU, a run fails instead of quietly falling back to the CPU.
* Every run reports the peak GPU memory and utilisation it reached.

**No limit on the number of images:** every image (or video frame) you give is used. If a run does not fit in GPU memory, the same process, with the models still loaded, finds the largest run that fits by measuring rather than guessing:
1. It runs a small probe.
2. It fits peak GPU memory against the number of views (or the resolution).
3. It runs the largest size that should fit, bisecting if the fit is off.
4. It keeps the best result that fitted.

By default it drops evenly spaced views (always keeping the first); you can choose to keep every view and lower the resolution instead, or to stop.

**Resolution:**
* The release's default is 756×756 px. Any multiples of 14 work, non-square included; the authors note that 1K and above is supported.
* Each image is scaled to cover the target size and then centre-cropped to it. A square target therefore crops the sides of a wide photo.
* That is why the cells below default to *match aspect*: the input's aspect ratio at about as many pixels as 756×756. A 16:9 video becomes 1008×560, for example.
* GPU memory and time grow with views × pixels.

> **License:** see [`InfiniSplat_v3/License.txt`](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v3/License.txt) and the repository's [Project Registration License (PRL) v1.0](https://github.com/Cortex78/InfiniSplat/blob/main/LICENSE). Academic research, education, evaluation and personal use are free without registration; project use by companies, universities or other organisations requires a free prior registration. Third-party components keep their own licenses.

## 1 · Setup

In [ ]:
#@title 1.1 · Check the runtime (GPU, RAM, disk)
import os, re, shutil, subprocess, sys
from pathlib import Path

query = None
if shutil.which('nvidia-smi'):
    query = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True)
if query is None or query.returncode != 0 or not query.stdout.strip():
    raise SystemExit('No NVIDIA GPU found. Choose Runtime > Change runtime type > A100 GPU, then run this cell again.')
gpu_name, vram_mib, compute_cap, driver = [s.strip() for s in query.stdout.strip().splitlines()[0].split(',')]
GPU_NAME, VRAM_GB, COMPUTE_CAP = gpu_name, float(vram_mib) / 1024, float(compute_cap)
# The newest CUDA runtime this driver supports, from the nvidia-smi banner; it picks the PyTorch build.
cuda_banner = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
DRIVER_CUDA = (int(cuda_banner[1]), int(cuda_banner[2])) if cuda_banner else (0, 0)
RAM_GB = int(Path('/proc/meminfo').read_text().split()[1]) / 1024**2
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
DISK_FREE_GB = shutil.disk_usage(BASE).free / 1024**3

print(f'GPU : {GPU_NAME}, {VRAM_GB:.0f} GB, compute capability {COMPUTE_CAP}, '
      f'driver {driver} (CUDA {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]})')
print(f'RAM : {RAM_GB:.1f} GB   disk free: {DISK_FREE_GB:.0f} GB   Python {sys.version.split()[0]}')
if 'A100' in GPU_NAME:
    print('A100 detected.')
else:
    print('Note: this notebook is set up for an A100 (Runtime > Change runtime type > A100 GPU).')
if COMPUTE_CAP < 8.0:
    print('Warning: v3 runs its transformer in bfloat16, which this GPU does not support natively; expect it to be '
          'slow or to fail.')
if RAM_GB < 24:
    print('Warning: building both models and loading 16 GB of weights needs plenty of system RAM; use an A100 or '
          'High-RAM runtime.')
if DISK_FREE_GB < 40:
    print('Warning: the environment, weights and results need about 30 GB of free disk.')

In [ ]:
#@title 1.2 · Get the code (InfiniSplat_v3 only)
REPO_URL = 'https://github.com/Cortex78/InfiniSplat'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}
#@markdown Only the `InfiniSplat_v3` folder is checked out (the repository also holds v1 and v2). If this branch does not exist, the repository's default branch is used.

REPO = BASE / 'InfiniSplat'
V3_DIR = REPO / 'InfiniSplat_v3'
if (REPO / '.git').is_dir():
    print(f'Reusing the existing clone at {REPO} (delete it to clone a different branch).')
else:
    clone = ['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse']
    if subprocess.run(clone + ['--branch', BRANCH, REPO_URL, str(REPO)]).returncode:
        print(f'Branch {BRANCH} was not found; cloning the default branch.')
        subprocess.run(clone + [REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'sparse-checkout', 'set', 'InfiniSplat_v3'], check=True)
if not (V3_DIR / 'inference.py').is_file():
    raise SystemExit(f'{V3_DIR} has no inference.py: this branch does not contain InfiniSplat v3.')
print(subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--format=%h %d %s (%cd)'],
                     capture_output=True, text=True).stdout)

In [ ]:
#@title 1.3 · Helper functions (run once; expand to read the code) { display-mode: "form" }
import codecs, json, shlex, socket, threading, time, urllib.parse, zipfile
import numpy as np
from PIL import Image, ImageOps

try:
    import plyfile  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plyfile'], check=True)

IN_COLAB = 'google.colab' in sys.modules
ENVS, OUTPUTS = BASE / 'envs', BASE / 'outputs'
V3_PY = ENVS / 'v3' / 'bin' / 'python'
V3_CKPTS, V3_HF_REPO = V3_DIR / 'checkpoints', 'ritianyu/SplatAnything'
V3_WEIGHTS = ('SplatAnything.ckpt', 'worldmirror2.safetensors')  # the names configs/inference.yaml expects
OUTPUTS.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}  # what v3 reads from images/
VIDEO_EXTS = {'.mp4', '.mov', '.m4v', '.webm', '.mkv', '.avi'}
OOM_CHOICES = {'fewer views': 'views', 'lower resolution': 'resolution', 'stop': 'stop'}
EXAMPLES = sorted(p.name for p in (V3_DIR / 'examples').iterdir() if (p / 'images').is_dir())


def env_with(*bin_dirs, **extra):
    """Environment overrides that put the given directories first on PATH."""
    return {'PATH': os.pathsep.join([*map(str, bin_dirs), os.environ['PATH']]), **extra}


# The v3 environment's python, node and splat-transform come first on PATH.
V3_ENV = env_with(ENVS / 'splat-v3' / 'node_modules' / '.bin', ENVS / 'v3' / 'bin',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')


def torch_index(choice):
    """PyTorch wheel tag and index: CUDA 12.8 (the tested builds) if the driver supports it, else 12.6."""
    tag = choice if choice != 'auto' else ('cu128' if DRIVER_CUDA >= (12, 8) else 'cu126')
    return tag, f'https://download.pytorch.org/whl/{tag}'


# Run inside an environment: fail unless PyTorch can compute on the GPU, and show what it gets.
CUDA_CHECK = r"""
import time, torch, torch.nn.functional as F
if not torch.cuda.is_available():
    raise SystemExit(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) cannot use the GPU. Check the "
                     "runtime type; with an old driver, rerun cell 1.4 with TORCH_CUDA = cu126.")
props = torch.cuda.get_device_properties(0)
x = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
x @ x
torch.cuda.synchronize()
start = time.time()
for _ in range(5):
    x @ x
torch.cuda.synchronize()
tflops = 5 * 2 * 8192**3 / (time.time() - start) / 1e12
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    q = torch.randn(1, 8, 2048, 64, device="cuda", dtype=torch.bfloat16)
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        F.scaled_dot_product_attention(q, q, q)
    flash = "yes"
except RuntimeError:
    flash = "no"
print(f"CUDA OK: torch {torch.__version__} (CUDA {torch.version.cuda}) on {props.name}, "
      f"{props.total_memory / 1024**3:.0f} GB, compute {props.major}.{props.minor}; "
      f"bf16 matmul {tflops:.0f} TFLOPS; FlashAttention: {flash}")
"""


def check_cuda(python):
    sh(f'{shlex.quote(str(python))} -c {shlex.quote(CUDA_CHECK)}')


class GpuMonitor:
    """Sample nvidia-smi while a run is in progress and report how much of the GPU it used."""

    def __init__(self, interval=0.25):
        self.interval, self.idle_gb, self.peak_gb, self.peak_util = interval, 0.0, 0.0, 0.0

    @staticmethod
    def sample():
        out = subprocess.run(['nvidia-smi', '-i', '0', '--query-gpu=memory.used,utilization.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
        used, util = out.strip().splitlines()[0].split(',')
        return float(used) / 1024, float(util)

    def __enter__(self):
        try:
            self.idle_gb = self.sample()[0]
        except (ValueError, IndexError, OSError):
            pass
        self.done = threading.Event()

        def loop():
            while True:
                try:
                    used, util = self.sample()
                    self.peak_gb, self.peak_util = max(self.peak_gb, used), max(self.peak_util, util)
                except (ValueError, IndexError, OSError):
                    pass
                if self.done.wait(self.interval):
                    break

        self.thread = threading.Thread(target=loop, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *exc):
        self.done.set()
        self.thread.join()
        print(f'GPU usage: {GPU_NAME}, peak memory {self.peak_gb:.1f} of {VRAM_GB:.0f} GB, '
              f'peak utilisation {self.peak_util:.0f}%')
        if exc[0] is None and self.peak_gb - self.idle_gb < 1:
            print('WARNING: this run barely used the GPU. Rerun the install cell and check its "CUDA OK" line.')


class CommandError(RuntimeError):
    def __init__(self, code, tail):
        low = tail.lower()
        self.code, self.oom = code, 'out of memory' in low
        if code in (-9, 137):
            hint = ('the process was killed, most likely for lack of system RAM while loading the weights; '
                    'use an A100 or High-RAM runtime, or restart the runtime to free memory.')
        elif 'out of memory' in low:
            hint = ('the GPU ran out of memory. Set ON_OOM (cell 2.1) to retry with fewer views or a lower '
                    'resolution, or ask for fewer views or pixels.')
        elif 'no space left' in low:
            hint = f'the disk is full. Delete old results in {OUTPUTS} or start a fresh runtime.'
        else:
            hint = 'see the output above.'
        super().__init__(f'Command failed with exit code {code}: {hint}')


def sh(cmd, cwd=None, env=None, show=None):
    """Run a bash command, stream its output, and raise CommandError if it fails."""
    print(f'$ {show or cmd}', flush=True)
    proc = subprocess.Popen(['bash', '-c', 'set -eo pipefail\n' + cmd], cwd=cwd,
                            env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder, tail = codecs.getincrementaldecoder('utf-8')('replace'), ''
    for chunk in iter(lambda: proc.stdout.read1(65536), b''):
        text = decoder.decode(chunk)
        sys.stdout.write(text)
        sys.stdout.flush()
        tail = (tail + text)[-20000:]
    code = proc.wait()
    if code:
        raise CommandError(code, tail)


def uv_venv(path, python):
    """Create a fresh virtualenv with uv, installing uv into the kernel first if needed."""
    if not shutil.which('uv'):
        sh(f'{shlex.quote(sys.executable)} -m pip install -q uv')
    shutil.rmtree(path, ignore_errors=True)
    sh(f'uv venv --python {python} {shlex.quote(str(path))}')


def require(path, cell):
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} is missing; run cell {cell} first.')


def pick_views(paths, count):
    """Indices of `count` evenly spaced items, keeping the first and last."""
    if count >= len(paths):
        return list(range(len(paths)))
    return sorted(set(np.linspace(0, len(paths) - 1, count).round().astype(int).tolist()))


def list_images(directory, exts=IMAGE_EXTS):
    return sorted((p for p in Path(directory).iterdir() if p.is_file() and p.suffix.lower() in exts),
                  key=lambda p: p.name)


def extract_zip(archive, dest, exts=IMAGE_EXTS):
    """Extract the images of a zip archive into dest, flattening folders into the file names."""
    with zipfile.ZipFile(archive) as zf:
        for member in sorted(zf.namelist()):
            path = Path(member)
            if path.suffix.lower() in exts and not path.name.startswith('.') and '__MACOSX' not in path.parts:
                (dest / '_'.join(path.parts)).write_bytes(zf.read(member))


def upload_files(kind):
    """Upload files through the browser into a new directory under OUTPUTS/uploads; zip archives are unpacked."""
    if not IN_COLAB:
        raise RuntimeError('Uploading needs Colab; set the path parameter instead.')
    from google.colab import files
    dest = OUTPUTS / 'uploads' / f'{kind}-{time.strftime("%Y%m%d-%H%M%S")}'
    dest.mkdir(parents=True)
    for name, data in files.upload().items():
        target = dest / Path(name).name
        target.write_bytes(data)
        Path(name).unlink(missing_ok=True)  # files.upload() also saves a copy in the working directory
        if target.suffix.lower() == '.zip':
            extract_zip(target, dest)
            target.unlink()
    return dest


def image_source(path, kind):
    """A folder of images from a folder path, a zip path, or (when empty) a browser upload."""
    if not path:
        return upload_files(kind)
    path = Path(path)
    if path.suffix.lower() == '.zip':
        dest = OUTPUTS / 'uploads' / f'{path.stem}-{time.strftime("%Y%m%d-%H%M%S")}'
        dest.mkdir(parents=True)
        extract_zip(path, dest)
        return dest
    return path


def sharpness(path):
    """Variance of the Laplacian of a downscaled grey image: higher is sharper."""
    with Image.open(path) as im:
        im.draft('L', (640, 640))
        im = im.convert('L')
        im.thumbnail((640, 640))
        g = np.asarray(im, np.float32)
    return float((4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]).var())


def extract_frames(video, count, start=0.0, end=0.0, candidates=3, max_side=2048):
    """Split the clip into `count` equal time slices and keep the sharpest of `candidates` frames in each."""
    video = Path(video)

    def probe(*entries):
        return json.loads(subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', *entries,
                                          '-of', 'json', str(video)], capture_output=True, text=True,
                                         check=True).stdout)
    info = probe('-show_entries', 'format=duration:stream=avg_frame_rate')
    num, _, den = info['streams'][0].get('avg_frame_rate', '0/0').partition('/')
    native_fps = float(num) / float(den) if float(den or 0) else 30.0
    try:
        duration = float(info['format']['duration'])
    except (KeyError, ValueError):  # e.g. browser-recorded WebM without a duration: count packets instead
        packets = probe('-count_packets', '-show_entries', 'stream=nb_read_packets')
        duration = int(packets['streams'][0]['nb_read_packets']) / native_fps
    end = min(end or duration, duration)
    if not 0 <= start < end:
        raise ValueError(f'Invalid time range {start}-{end} s for a {duration:.1f} s video.')
    out = OUTPUTS / 'video_frames' / f'{video.stem}-{time.strftime("%Y%m%d-%H%M%S")}'
    pool, frames = out / 'candidates', out / 'frames'
    pool.mkdir(parents=True)
    frames.mkdir()
    # ffmpeg applies the video's rotation metadata; frames are capped at max_side pixels on the long side.
    scale = f"scale='if(gt(iw,ih),min({max_side},iw),-2)':'if(gt(iw,ih),-2,min({max_side},ih))'"
    sh(f'ffmpeg -hide_banner -loglevel error -ss {start} -t {end - start} -i {shlex.quote(str(video))} '
       f'-vf "fps={min(count * candidates / (end - start), native_fps):.6f},{scale}" -q:v 2 '
       f'{shlex.quote(str(pool))}/%06d.jpg')
    pool_files = sorted(pool.iterdir())
    if len(pool_files) < 2:
        raise ValueError('Fewer than two frames could be read from the video.')
    for i, group in enumerate(np.array_split(np.arange(len(pool_files)), min(count, len(pool_files)))):
        best = max(group, key=lambda j: sharpness(pool_files[j]))
        pool_files[best].rename(frames / f'frame_{i:05d}.jpg')
    shutil.rmtree(pool)
    kept = list_images(frames)
    print(f'Kept the sharpest frame of each of {len(kept)} time slices from {end - start:.1f} s of video '
          f'({len(pool_files)} candidates).')
    import matplotlib.pyplot as plt
    shown = [kept[i] for i in pick_views(kept, 8)]
    fig, axes = plt.subplots(1, len(shown), figsize=(2.2 * len(shown), 2))
    for ax, path in zip(np.atleast_1d(axes), shown):
        with Image.open(path) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(path.stem, fontsize=8)
        ax.axis('off')
    plt.show()
    return frames


# ---------------------------------------------------------------- running v3
# Runs inside the v3 environment through the repository's own inference.py argument parser and config checks,
# then loads both models once. If every view does not fit in GPU memory, a small probe measures the real peak
# memory, a linear fit picks the size to try, bisection corrects the fit, and the best result that fitted is
# exported with the repository's own export functions.
RUNNER = r"""
import argparse, functools, gc, json, sys, time
from pathlib import Path
import numpy as np
import torch

# The notebook's extra settings come first; everything else goes to the repository's own inference.py parser
# and config checks, so the release's validation applies unchanged.
parser = argparse.ArgumentParser(allow_abbrev=False)
parser.add_argument('--repo', required=True)  # the InfiniSplat_v3 directory
parser.add_argument('--on-oom', choices=['views', 'resolution', 'stop'], default='views')
parser.add_argument('--spacing-target', type=float, default=0.0013227513)  # GaussianWorldMirror default
parser.add_argument('--decode-chunk', type=int, default=8)  # views per Gaussian-decoder chunk
parser.add_argument('--depth-chunk', type=int, default=8)  # views per depth-head (DPT) chunk
parser.add_argument('--flying-rtol', type=float, default=0.03)  # flying_point_mask defaults
parser.add_argument('--flying-kernel', type=int, default=3)
parser.add_argument('--sky-threshold', type=float, default=0.5)  # SkySegmenter prob_thresh
parser.add_argument('--export-scale-quantile', type=float, default=0.0)  # 0 = export every Gaussian (release)
parser.add_argument('--tf32', action='store_true')
known, rest = parser.parse_known_args()
sys.path.insert(0, known.repo)
import inference
from omegaconf import OmegaConf
import src.models.models.rasterization as rasterization
from src.models.models.gaussian_worldmirror import GaussianWorldMirror
from src.models.models.geometry_worldmirror import GeometryWorldMirror
from src.utils.checkpoints import load_model_weights
from src.utils.point_filter import flying_point_mask
from src.utils.scene_io import build_views, export_gaussians, export_points, export_depth_rgb
from src.utils.sky_segmentation import SkySegmenter

args = inference.parse_args(rest)
cfg = inference.load_config(args)
device = torch.device(args.device)
if device.type != 'cuda' or not torch.cuda.is_available():
    raise SystemExit('CUDA is unavailable: this notebook runs v3 on the GPU only.')
torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = known.tf32
pose = bool(cfg.wrapper.enable_cond)
cond_flags = [1, 0, 1] if pose else [0, 0, 0]
rasterization.flying_point_mask = functools.partial(flying_point_mask, rtol=known.flying_rtol,
                                                    kernel_size=known.flying_kernel)
segmenter = (SkySegmenter(args.sky_onnx_path, device=args.device, prob_thresh=known.sky_threshold)
             if args.sky_seg else None)
model = GaussianWorldMirror(**OmegaConf.to_container(cfg.wrapper.model), enable_cond=pose,
                            filter_sky_infer=args.sky_seg, spacing_target=known.spacing_target,
                            decode_chunk_size=known.decode_chunk)
teacher = GeometryWorldMirror(enable_cond=pose)
load_model_weights(model, cfg.wrapper.SplatAnything_pretrained)
load_model_weights(teacher, cfg.wrapper.worldmirror2_pretrained)
model.to(device).eval().requires_grad_(False)
teacher.to(device).eval().requires_grad_(False)
teacher.depth_head.forward = functools.partial(teacher.depth_head.forward, frames_chunk_size=known.depth_chunk)

GiB = 2**30
capacity = torch.cuda.get_device_properties(device).total_memory
weights = torch.cuda.memory_allocated(device)
budget = 0.92 * capacity  # room for the CUDA context, cuBLAS workspaces and fragmentation
W0, H0 = args.resolution
cache = {}


def views_at(width):  # preprocessed views of every image on the CPU, at `width` and the matching height
    height = max(14, round(H0 * width / W0 / 14) * 14)
    if width not in cache:
        cache.clear()
        cache[width] = build_views(args.images_path, args.cameras, (width, height), 'cpu',
                                   cam_align=not args.no_cam_align, sky_segmenter=segmenter,
                                   sky_max_fraction=args.sky_max_fraction) + ((width, height),)
    return cache[width]


views_all, scene_name, img_paths, _ = views_at(W0)
total = len(img_paths)
# The size being searched: the number of views, or the width (memory grows with its square at a fixed aspect).
power = 2 if known.on_oom == 'resolution' else 1
full, low = (W0, 112) if power == 2 else (total, 2)
attempts = []
print(f'{total} images at {W0}x{H0} px, {"pose-conditioned" if pose else "RGB-only"}; the model weights use '
      f'{weights / GiB:.1f} of {capacity / GiB:.0f} GB of GPU memory', flush=True)


def snap(x):  # whole views, or widths on the model's 14-pixel patch grid
    return max(low, int(x) // 14 * 14 if power == 2 else int(x))


def to_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(to_cpu(item) for item in value)
    return value


EXPORT_KEYS = ('splats', 'points_ply_means', 'points_ply_rgb', 'depth', 'export_camera_poses', 'export_camera_intrs')


def attempt(x):
    views_cpu, _, paths, size = views_at(x) if power == 2 else views_at(W0)
    count = total if power == 2 else x
    # Evenly spaced views that always keep view 0, which anchors the camera frame.
    selected = list(range(total)) if count >= total else sorted(
        set(np.linspace(0, total - 1, count).round().astype(int).tolist()))
    print(f'Attempt {len(attempts) + 1}: {len(selected)} of {total} views at {size[0]}x{size[1]} px', flush=True)
    index = torch.tensor(selected)
    subset = {key: value.index_select(1, index) for key, value in views_cpu.items()}
    torch.cuda.reset_peak_memory_stats(device)
    start, result, error = time.time(), None, None
    views = geometry = preds = None
    try:
        views = {key: value.to(device) for key, value in subset.items()}
        with torch.inference_mode():
            geometry = teacher(views, cond_flags=cond_flags)
            preds = model(views, geometry, cond_flags=cond_flags)
        torch.cuda.synchronize(device)
        # Keep only what the exports need, on the CPU, so the GPU is free for a larger attempt.
        result = to_cpu({key: preds[key] for key in EXPORT_KEYS if key in preds})
    except RuntimeError as exc:  # includes torch.OutOfMemoryError
        if 'out of memory' not in str(exc).lower() and 'CUBLAS_STATUS_ALLOC_FAILED' not in str(exc):
            raise
        error = str(exc).splitlines()[0]
    views = geometry = preds = None  # release the GPU tensors before emptying the cache
    peak = torch.cuda.max_memory_allocated(device)
    gc.collect()
    torch.cuda.empty_cache()
    attempts.append(dict(views=len(selected), resolution=list(size), seconds=round(time.time() - start, 1),
                         peak_gb=round(peak / GiB, 2), out_of_memory=error is not None))
    print(f'  {"out of GPU memory" if error else "fits"}: peak {peak / GiB:.1f} of {capacity / GiB:.0f} GB, '
          f'{attempts[-1]["seconds"]} s', flush=True)
    return result, peak, error, selected, size, subset, [paths[i] for i in selected]


def predict(measured):
    # Largest size whose peak should stay within 95% of the budget: a line through the last two successful
    # (size**power, peak) points, or through the weights alone at size 0 after the first success.
    (x1, p1), (x2, p2) = measured[-2] if len(measured) > 1 else (0, weights), measured[-1]
    slope = max((p2 - p1) / max(x2 ** power - x1 ** power, 1), 1.0)
    return max(0.0, (0.95 * budget - (p2 - slope * x2 ** power)) / slope) ** (1 / power)


measured, best, failed, x = [], None, None, full
while len(attempts) < 12:
    result, peak, error, selected, size, subset, paths = attempt(x)
    if result is not None:
        best = (x, result, selected, size, subset, paths)
        measured.append((x, peak))
        if x == full:
            break
        target = snap(predict(measured))
        if failed is not None:
            target = min(target, failed - (14 if power == 2 else 1))
        if target <= x or (power == 1 and target < 1.05 * x):  # not worth another attempt
            break
        x = target
        continue
    if known.on_oom == 'stop':
        raise SystemExit(f'GPU out of memory with {len(selected)} views at {size[0]}x{size[1]} px: {error}')
    failed = x
    if best is not None:  # bisect between the largest size that fitted and the smallest that did not
        x = snap((best[0] + failed) / 2)
        if x <= best[0]:
            break
    elif x <= low:
        raise SystemExit(f'Does not fit in GPU memory even with {len(selected)} views at {size[0]}x{size[1]} px: '
                         f'{error}')
    else:  # a small probe that fits gives the real memory use to extrapolate from
        x = min(snap(x / 4 if power == 1 else x / 2), failed - 1)
if best is None:
    raise SystemExit('No attempt fitted in GPU memory.')

x, preds, selected, size, subset, paths = best
print(f'Using {len(selected)} of {total} views at {size[0]}x{size[1]} px after {len(attempts)} attempt(s).',
      flush=True)
# The same checks and exports as inference.py.
outdir = Path(args.output_path) / scene_name
outdir.mkdir(parents=True, exist_ok=True)
if not len(preds['splats']['means'][0]):
    raise SystemExit('All Gaussians were filtered. Check the input, sky masks, and scale thresholds.')
if not (preds['splats']['opacities'][0] >= cfg.wrapper.infer_opacity_thresh).any():
    raise SystemExit('All Gaussians are below the opacity threshold. Lower it and check the checkpoint.')
if known.export_scale_quantile:
    # Drop the largest-scale tail like save_gs_ply's quantile_threshold, among the Gaussians that pass the
    # opacity threshold, but with NumPy: torch.quantile refuses inputs above 16M elements.
    splats = preds['splats']
    biggest = splats['scales'][0].amax(dim=-1)
    passing = splats['opacities'][0].reshape(-1) >= cfg.wrapper.infer_opacity_thresh
    limit = float(np.quantile(biggest[passing].numpy(), known.export_scale_quantile))
    keep = biggest <= limit
    print(f'Export scale quantile {known.export_scale_quantile}: dropping {int((~keep).sum()):,} Gaussians '
          f'larger than {limit:.4g}', flush=True)
    for key in splats:
        splats[key][0] = splats[key][0][keep]
export_gaussians(preds, outdir, cfg.wrapper.infer_opacity_thresh, None)
export_points(preds, outdir)
if not args.no_depth_rgb:
    export_depth_rgb(preds, subset, paths, outdir)
cameras = {'extrinsics': [], 'intrinsics': []}
for i, image in enumerate(paths):
    for name, key in [('extrinsics', 'export_camera_poses'), ('intrinsics', 'export_camera_intrs')]:
        cameras[name].append({'camera_id': Path(image).stem, 'matrix': preds[key][0, i].tolist()})
(outdir / 'cameras.json').write_text(json.dumps(cameras, indent=2) + '\n')
args.resolution = list(size)
OmegaConf.save(cfg, outdir / 'config.yaml', resolve=True)
(outdir / 'run.json').write_text(json.dumps({**vars(args), 'images': paths, 'cond_flags': cond_flags,
                                             'notebook_settings': vars(known)}, indent=2) + '\n')
(outdir / 'notebook_run.json').write_text(json.dumps(
    dict(attempts=attempts, used=paths, inputs=total, resolution=list(size)), indent=2) + '\n')
print(f'[Done] {outdir}')
"""

# Release values of every setting in cell 2.1: configs/inference.yaml, then code defaults of inference.py,
# GaussianWorldMirror, flying_point_mask and SkySegmenter. Cell 2.1 replaces SETTINGS.
DEFAULT_SETTINGS = dict(
    voxel_size=0.002, filter_large_scales=True, max_scale_pct=99.8, max_scale_abs=0.0,
    filter_flying_points=False, flying_rtol=0.03, flying_kernel=3,
    opacity_threshold=0.05, export_scale_quantile=0.0, export_depth_rgb=True,
    sky_removal=False, sky_threshold=0.5, sky_max_fraction=0.95,
    keep_camera_frame=False, spacing_target=0.0013227513,
    on_oom='views', decode_chunk=8, depth_chunk=8, tf32=False,
    export_html=True, viewer_max_splats=6_000_000)
SETTINGS = dict(DEFAULT_SETTINGS)
LAST_INPUT = {}  # what the latest run was given, for cell 3.6


def check_release_defaults():
    """Warn if the checked-out configs/inference.yaml no longer matches the defaults above."""
    import yaml
    wrapper = yaml.safe_load((V3_DIR / 'configs' / 'inference.yaml').read_text())['wrapper']
    model = wrapper['model']
    release = dict(voxel_size=model['gs_voxel_size'], filter_large_scales=model['filter_large_scale_infer'],
                   max_scale_pct=model['max_scale_pct'] or 0.0, max_scale_abs=model['max_scale_abs'] or 0.0,
                   filter_flying_points=model['filter_flying_points_infer'],
                   opacity_threshold=wrapper['infer_opacity_thresh'])
    changed = {k: v for k, v in release.items() if v != DEFAULT_SETTINGS[k]}
    if changed:
        print(f'Note: this checkout\'s configs/inference.yaml has different defaults: {changed}. '
              'Cell 2.1 uses its own values; set them to match if you want the release behaviour.')


def snap14(value):
    return max(14, round(value / 14) * 14)


def resolve_size(images, resolution, width=0, height=0, release=(756, 756)):
    """Model input size (multiples of 14) for one of the resolution choices of the cells below."""
    if resolution.startswith('custom'):
        if not (width > 0 and height > 0):
            raise ValueError('Set WIDTH and HEIGHT (in pixels) for a custom resolution.')
        size = (snap14(width), snap14(height))
        if size != (width, height):
            print(f'{width}x{height} is not a multiple of 14; using {size[0]}x{size[1]}.')
        return size
    if resolution.startswith('release'):
        return release
    side = int(re.search(r'(\d+)', resolution)[1])
    if resolution.startswith('square'):
        return side, side
    with Image.open(images[0]) as im:
        w, h = im.size  # v3 reads the stored pixels, without applying EXIF rotation
    aspect = w / h  # same pixel count as side x side, at the input's aspect ratio
    return snap14(side * aspect ** 0.5), snap14(side / aspect ** 0.5)


def make_scene(images, name, cameras_json=None, upright=False):
    """A v3 scene directory (images/ plus an optional prior_camera.json) linking to the given images.

    With `upright`, photos whose EXIF data says they are rotated are stored upright (v3 itself ignores EXIF
    orientation); never with cameras, whose intrinsics refer to the stored pixels.
    """
    scene = OUTPUTS / 'scenes' / f'{name}-{time.strftime("%Y%m%d-%H%M%S")}'
    (scene / 'images').mkdir(parents=True)
    rotated = 0
    for path in images:
        path = Path(path).resolve()
        target = scene / 'images' / path.name
        orientation = None
        if upright:
            with Image.open(path) as im:
                orientation = im.getexif().get(0x0112)
                if orientation not in (None, 1):
                    ImageOps.exif_transpose(im).convert('RGB').save(target.with_suffix('.png'))
                    rotated += 1
        if orientation in (None, 1):
            target.symlink_to(path)
    if rotated:
        print(f'{rotated} photo(s) were stored rotated (EXIF); they are passed to v3 upright.')
    if cameras_json:
        shutil.copy(cameras_json, scene / 'prior_camera.json')
    return scene


def cameras_from_colmap(model_dir, scene):
    """prior_camera.json for a scene from an undistorted PINHOLE/SIMPLE_PINHOLE COLMAP model."""
    sh(shlex.join(map(str, [V3_PY, V3_DIR / 'scripts' / 'convert_cameras.py', 'colmap', '--model',
                            Path(model_dir).resolve(), '--images', scene / 'images',
                            '--output', scene / 'prior_camera.json'])), env=V3_ENV)


def write_config(path, s):
    """configs/inference.yaml with the settings of cell 2.1 (same keys, so the release's checks still apply)."""
    import yaml
    cfg = yaml.safe_load((V3_DIR / 'configs' / 'inference.yaml').read_text())
    cfg['wrapper']['infer_opacity_thresh'] = float(s['opacity_threshold'])
    cfg['wrapper']['model'].update(
        gs_voxel_size=float(s['voxel_size']), filter_large_scale_infer=bool(s['filter_large_scales']),
        max_scale_pct=float(s['max_scale_pct']) or None, max_scale_abs=float(s['max_scale_abs']) or None,
        filter_flying_points_infer=bool(s['filter_flying_points']))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))


def run_v3(scene, mode='rgb', size=(756, 756), settings=None):
    """Run v3 on a scene directory with the given (or cell 2.1's) settings; return the output directory."""
    s = {**DEFAULT_SETTINGS, **(settings or SETTINGS)}
    scene = Path(scene)
    require(V3_PY, '1.4')
    for name in V3_WEIGHTS:
        require(V3_CKPTS / name, '1.5')
    if s['sky_removal']:
        require(V3_CKPTS / 'skyseg.onnx', '1.5 with DOWNLOAD_SKY_MODEL')
    if mode == 'pose' and not (scene / 'prior_camera.json').is_file():
        raise ValueError('pose mode needs cameras (prior_camera.json); use mode rgb to predict them instead.')
    runner, config = ENVS / 'v3_runner.py', OUTPUTS / 'configs' / f'{scene.name}.yaml'
    runner.write_text(RUNNER)
    write_config(config, s)
    argv = [V3_PY, runner, '--repo', V3_DIR, '--on-oom', s['on_oom'], '--spacing-target', s['spacing_target'],
            '--decode-chunk', s['decode_chunk'], '--depth-chunk', s['depth_chunk'],
            '--flying-rtol', s['flying_rtol'], '--flying-kernel', s['flying_kernel'],
            '--sky-threshold', s['sky_threshold'], '--export-scale-quantile', s['export_scale_quantile'],
            '--config_path', config, '--input_path', scene, '--mode', mode, '--output_path', OUTPUTS / 'runs',
            '--resolution', size[0], size[1], '--opacity_thresh', s['opacity_threshold'],
            '--sky_max_fraction', s['sky_max_fraction'], '--device', 'cuda']
    argv += (['--tf32'] * bool(s['tf32']) + ['--sky_seg'] * bool(s['sky_removal'])
             + ['--no_cam_align'] * bool(s['keep_camera_frame']) + ['--no_depth_rgb'] * (not s['export_depth_rgb']))
    start = time.time()
    with GpuMonitor():
        sh(shlex.join(map(str, argv)), cwd=V3_DIR, env=V3_ENV)
    out = OUTPUTS / 'runs' / scene.name
    run = json.loads((out / 'notebook_run.json').read_text())
    print(f'Finished in {time.time() - start:.0f} s ({len(run["attempts"])} attempt(s)): {out}')
    if s['export_html']:
        make_viewer(out, s['viewer_max_splats'])
    return out


def reconstruct(images, name, mode='rgb', cameras=None, num_views=0, resolution='match aspect, 756² px',
                width=0, height=0, release=(756, 756), settings=None):
    """Run v3 on every image, or on an evenly spaced `num_views` of them (always keeping the first).

    `cameras` is a prior_camera.json or a COLMAP model folder (pose mode). Returns the output directory.
    """
    LAST_INPUT.clear()
    LAST_INPUT.update(images=list(images), name=name, mode=mode, cameras=cameras, num_views=num_views,
                      resolution=resolution, width=width, height=height, release=release)
    images = [Path(p) for p in images]
    images = [images[i] for i in pick_views(images, num_views or len(images))]
    camera_file = cameras if cameras and Path(cameras).is_file() else None
    scene = make_scene(images, f'{name}-{mode}', cameras_json=camera_file, upright=mode == 'rgb')
    if cameras and Path(cameras).is_dir():  # a COLMAP sparse model
        cameras_from_colmap(cameras, scene)
    size = resolve_size(list_images(scene / 'images'), resolution, width, height, release)
    names = [p.name for p in images]
    shown = ', '.join(names) if len(names) <= 8 else ', '.join(names[:3]) + ', …, ' + names[-1]
    print(f'{mode} mode, {len(images)} views at {size[0]}x{size[1]} px: {shown}')
    return run_v3(scene, mode=mode, size=size, settings=settings)


def make_viewer(out, max_splats=6_000_000):
    """scene.html (or viewer/scene_preview.html for very large scenes) from gaussians.ply with splat-transform.

    The cloud is in OpenCV axes (y down); the viewer turns it upright and starts at the first input camera.
    """
    from plyfile import PlyData, PlyElement
    ply, html = out / 'gaussians.ply', out / 'scene.html'
    vertices = PlyData.read(str(ply))['vertex'].data
    xyz = np.stack([vertices['x'], vertices['y'], vertices['z']], 1).astype(np.float64)
    if len(vertices) > max_splats:
        (out / 'viewer').mkdir(exist_ok=True)
        keep = np.sort(np.random.default_rng(0).choice(len(vertices), max_splats, replace=False))
        ply, html = out / 'viewer' / 'scene_preview.ply', out / 'viewer' / 'scene_preview.html'
        PlyData([PlyElement.describe(vertices[keep], 'vertex')]).write(str(ply))
        print(f'{len(vertices):,} Gaussians: the viewer shows a random {max_splats:,} of them; '
              'gaussians.ply keeps them all.')
    del vertices
    cams = json.loads((out / 'cameras.json').read_text())
    c2w = np.asarray(cams['extrinsics'][0]['matrix'], np.float64)
    K = np.asarray(cams['intrinsics'][0]['matrix'], np.float64)
    local = (xyz - c2w[:3, 3]) @ c2w[:3, :3]
    ahead = local[local[:, 2] > 0, 2]
    depth = float(np.median(ahead)) if len(ahead) else 1.0
    flip = np.diag([1.0, -1.0, -1.0])  # what `-r 180,0,0` does to every Gaussian
    position = flip @ (c2w[:3, 3] - 0.05 * depth * c2w[:3, 2])
    target = flip @ (c2w[:3, 3] + depth * c2w[:3, 2])
    fov = float(np.degrees(2 * np.arctan(K[1, 2] / K[1, 1])))  # vertical field of view of the input camera
    settings = dict(version=2, tonemapping='none', highPrecisionRendering=False,
                    background=dict(color=[0.4, 0.4, 0.4]), animTracks=[], annotations=[], startMode='default',
                    cameras=[dict(initial=dict(position=position.tolist(), target=target.tolist(), fov=fov))])
    settings_path = out / 'viewer_settings.json'
    settings_path.write_text(json.dumps(settings))
    sh(shlex.join(['splat-transform', '-w', str(ply), '-r', '180,0,0', '-H', '0', '-E', str(settings_path),
                   '--max-workers', str(os.cpu_count() or 2), str(html)]), env=V3_ENV)
    return html


# ---------------------------------------------------------------- looking at Gaussians
SH_C0 = 0.28209479177387814


def load_gaussians(ply_path):
    """Means, display colours, opacities and single-property metadata elements of a Gaussian PLY."""
    from plyfile import PlyData
    ply = PlyData.read(str(ply_path))
    v = ply['vertex'].data
    xyz = np.stack([v['x'], v['y'], v['z']], 1).astype(np.float64)
    rgb = np.clip(0.5 + SH_C0 * np.stack([v[f'f_dc_{i}'] for i in range(3)], 1), 0, 1).astype(np.float32)
    alpha = 1 / (1 + np.exp(-v['opacity'].astype(np.float64)))
    extra = {e.name: np.asarray(e.data[e.properties[0].name]) for e in ply.elements
             if e.name != 'vertex' and len(e.properties) == 1}
    return xyz, rgb, alpha, extra


def render_points(xyz, rgb, alpha, c2w, K, size, min_alpha=0.1):
    """Nearest-point z-buffer projection of Gaussian centres (OpenCV camera). A quick preview, not a splat render."""
    W, H = size
    w2c = np.linalg.inv(c2w)
    keep = alpha > min_alpha
    p = xyz[keep] @ w2c[:3, :3].T + w2c[:3, 3]
    c = rgb[keep]
    front = p[:, 2] > 1e-6
    p, c = p[front], c[front]
    u = np.floor(K[0, 0] * p[:, 0] / p[:, 2] + K[0, 2] + 0.5).astype(np.int64)
    v = np.floor(K[1, 1] * p[:, 1] / p[:, 2] + K[1, 2] + 0.5).astype(np.int64)
    inside = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    pixel, depth, c = v[inside] * W + u[inside], p[inside, 2], c[inside]
    order = np.lexsort((depth, pixel))  # group by pixel, nearest first
    pixel, c = pixel[order], c[order]
    first = np.r_[True, pixel[1:] != pixel[:-1]]
    image = np.full((H * W, 3), 0.4, np.float32)  # the HTML viewer's grey background
    image[pixel[first]] = c[first]
    return image.reshape(H, W, 3)


def orbit(c2w, pivot_depth, degrees):
    """Yaw a camera about the point `pivot_depth` in front of it."""
    t = np.radians(degrees)
    R = np.array([[np.cos(t), 0, np.sin(t)], [0, 1, 0], [-np.sin(t), 0, np.cos(t)]])
    pivot = np.array([0.0, 0.0, pivot_depth])
    local = np.eye(4)
    local[:3, :3], local[:3, 3] = R, pivot - R @ pivot
    return c2w @ local


def preview(gaussians, c2w, K, size, reference=None, title='', degrees=12, max_side=640):
    """Input image next to point projections from the input camera and two orbiting novel views."""
    import matplotlib.pyplot as plt
    xyz, rgb, alpha = gaussians
    scale = min(1.0, max_side / max(size))
    K = np.diag([scale, scale, 1.0]) @ K
    size = (max(1, round(size[0] * scale)), max(1, round(size[1] * scale)))
    cam = (xyz - c2w[:3, 3]) @ c2w[:3, :3]
    visible = (alpha > 0.1) & (cam[:, 2] > 1e-6)
    pivot = float(np.median(cam[visible, 2])) if visible.any() else 1.0
    panels = [] if reference is None else [('Input image', reference)]
    for label, deg in ((f'{-degrees}° orbit', -degrees), ('Input camera', 0), (f'+{degrees}° orbit', degrees)):
        panels.append((label, render_points(xyz, rgb, alpha, orbit(c2w, pivot, deg), K, size)))
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4 * size[1] / size[0] + 0.7))
    for ax, (label, image) in zip(np.atleast_1d(axes), panels):
        ax.imshow(image)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(f'{title}\nGaussian centres projected as points (holes are expected; '
                 'the HTML viewer below renders the real splats)', fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_3d(gaussians, title='', max_points=60_000):
    """Interactive point cloud of a random subset of Gaussian centres."""
    import plotly.graph_objects as go
    xyz, rgb, alpha = gaussians
    keep = np.flatnonzero(alpha > 0.1)
    if len(keep) > max_points:
        keep = np.random.default_rng(0).choice(keep, max_points, replace=False)
    p, c = xyz[keep], (rgb[keep] * 255).astype(np.uint8)
    lo, hi = np.percentile(p, [1, 99], axis=0)
    margin = (hi - lo) * 0.1
    inside = np.all((p >= lo - margin) & (p <= hi + margin), axis=1)  # drop far floaters from the view box
    p, c = p[inside], c[inside]
    hidden = dict(visible=False)
    fig = go.Figure(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers', hoverinfo='skip',
                                 marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c])))
    fig.update_layout(title=f'{title} ({len(p):,} of {len(xyz):,} Gaussians, drag to rotate)',
                      height=600, margin=dict(l=0, r=0, t=40, b=0), paper_bgcolor='#666666',
                      font=dict(color='white'),
                      scene=dict(aspectmode='data', xaxis=hidden, yaxis=hidden, zaxis=hidden, bgcolor='#666666',
                                 camera=dict(up=dict(x=0, y=-1, z=0), eye=dict(x=0, y=-0.4, z=-1.8))))
    fig.show()


def show_depths(out, count=4):
    """Processed input images above their predicted depth maps, for a few evenly spaced views."""
    import matplotlib.pyplot as plt
    rgbs = sorted((Path(out) / 'rgb').glob('*.png'))
    if not rgbs:
        return
    rgbs = [rgbs[i] for i in pick_views(rgbs, count)]
    fig, axes = plt.subplots(2, len(rgbs), figsize=(3.2 * len(rgbs), 6.4), squeeze=False)
    for col, rgb in enumerate(rgbs):
        depth = rgb.parent.parent / 'depth' / rgb.name
        for row, (path, label) in enumerate(((rgb, rgb.stem), (depth, 'predicted depth'))):
            axes[row][col].axis('off')
            if path.is_file():
                axes[row][col].imshow(Image.open(path), cmap='gray')
                axes[row][col].set_title(label, fontsize=9)
    plt.tight_layout()
    plt.show()


def show_result(out, html=True):
    """Summary, point previews, depth maps, 3D plot and HTML viewer for one output directory."""
    out = Path(out)
    run = json.loads((out / 'notebook_run.json').read_text())
    info = json.loads((out / 'run.json').read_text())
    cams = json.loads((out / 'cameras.json').read_text())
    width, height = run['resolution']
    mode = 'pose' if info['cond_flags'][0] else 'rgb'
    xyz, rgb, alpha, _ = load_gaussians(out / 'gaussians.ply')
    print(f'{mode} mode, {len(run["used"])} of {run["inputs"]} views at {width}x{height} px, {len(xyz):,} Gaussians')
    for a in run['attempts']:
        print(f'  attempt: {a["views"]} views at {a["resolution"][0]}x{a["resolution"][1]} px, peak '
              f'{a["peak_gb"]} GB, {a["seconds"]} s{" (out of GPU memory)" if a["out_of_memory"] else ""}')
    list_files(out)
    c2w = np.asarray(cams['extrinsics'][0]['matrix'], np.float64)
    K = np.asarray(cams['intrinsics'][0]['matrix'], np.float64)
    first = out / 'rgb' / f'{cams["extrinsics"][0]["camera_id"]}.png'
    reference = np.asarray(Image.open(first).convert('RGB')) if first.is_file() else None
    preview((xyz, rgb, alpha), c2w, K, (width, height), reference, title=out.name)
    show_depths(out)
    plot_3d((xyz, rgb, alpha), title=out.name)
    if html:
        preview_html = out / 'viewer' / 'scene_preview.html'
        show_viewer(preview_html if preview_html.is_file() else out / 'scene.html')


def list_files(directory, per_folder=8):
    """File sizes in a directory; folders with many files (such as per-view depth maps) are summarised."""
    directory = Path(directory)
    for path in sorted(directory.iterdir()):
        if path.name.startswith('.'):
            continue
        files = [path] if path.is_file() else sorted(p for p in path.rglob('*') if p.is_file())
        if path.is_dir() and len(files) > per_folder:
            print(f'  {path.name + "/":40s} {sum(p.stat().st_size for p in files) / 1e6:8.1f} MB in {len(files)} files')
            continue
        for p in files:
            print(f'  {p.relative_to(directory)!s:40s} {p.stat().st_size / 1e6:8.1f} MB')


# ---------------------------------------------------------------- viewers and downloads
VIEWER_PORT = 8765


def ensure_file_server():
    """Serve OUTPUTS over HTTP so Colab can proxy the HTML viewers and videos into the notebook."""
    def is_up():
        with socket.socket() as s:
            return s.connect_ex(('127.0.0.1', VIEWER_PORT)) == 0
    if is_up():
        return
    subprocess.Popen([sys.executable, '-m', 'http.server', str(VIEWER_PORT), '--directory', str(OUTPUTS)],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(100):
        time.sleep(0.1)
        if is_up():
            return
    raise RuntimeError('Could not start the local file server for the viewer.')


def served_path(path):
    return '/' + urllib.parse.quote(Path(path).resolve().relative_to(OUTPUTS.resolve()).as_posix())


def show_viewer(html_path, height=640):
    html_path = Path(html_path)
    if not html_path.is_file():
        print('No HTML viewer for this run (it was disabled, or splat-transform was unavailable).')
        return
    if not IN_COLAB:
        print('Open this file in a browser:', html_path)
        return
    from google.colab import output
    ensure_file_server()
    print(f'Interactive viewer ({html_path.stat().st_size / 1e6:.0f} MB; it can take a few seconds to load). '
          'Drag to orbit, scroll to zoom.')
    output.serve_kernel_port_as_iframe(VIEWER_PORT, path=served_path(html_path), height=height)
    output.serve_kernel_port_as_window(VIEWER_PORT, path=served_path(html_path),
                                       anchor_text='Open the viewer in a new browser tab')


def download(path):
    """Download a file, or a directory as a zip, through the browser."""
    path = Path(path)
    if path.is_dir():
        path = Path(shutil.make_archive(str(path), 'zip', root_dir=path))
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print('Saved at', path)


check_release_defaults()
print(f'Helpers ready. Examples: {", ".join(EXAMPLES)}.')

In [ ]:
#@title 1.4 · Install the v3 environment with CUDA PyTorch (about 3–5 min)
TORCH_CUDA = 'auto'  #@param ["auto", "cu128", "cu126"]
#@markdown `auto` installs the CUDA 12.8 build of PyTorch 2.7.1 when the driver supports it, otherwise the CUDA 12.6 build. v3 needs PyTorch 2.4 or newer and has no compiled extensions. The cell ends with a `CUDA OK` line from a bf16 matmul on the GPU; it fails if PyTorch cannot use the GPU.

# Follows InfiniSplat_v3/README.md (Python 3.10+, PyTorch 2.4+, then requirements.txt) in an isolated env, so the
# notebook kernel's packages are never touched (v3 pins numpy<2 and opencv<4.12).
tag, index = torch_index(TORCH_CUDA)
marker = ENVS / 'v3' / '.ready'
if not marker.is_file() or marker.read_text() != tag:
    uv_venv(ENVS / 'v3', '3.12')
    pip = f'uv pip install --python {V3_PY}'
    sh(f'{pip} torch==2.7.1 --index-url {index}')
    # Plus the Hugging Face CLI for the weights and node for the HTML viewer converter.
    sh(f'{pip} -r requirements.txt "huggingface_hub>=0.34" nodejs-wheel==22.20.0', cwd=V3_DIR)
    # splat-transform turns gaussians.ply into a standalone HTML viewer.
    sh(f'npm install --prefix {ENVS / "splat-v3"} --no-audit --no-fund @playcanvas/splat-transform@2.6.0', env=V3_ENV)
    marker.write_text(tag)
check_cuda(V3_PY)
sh(f'{V3_PY} inference.py --help | sed -n 1,3p; {V3_PY} -c "from src.models.models.gaussian_worldmirror import '
   'GaussianWorldMirror; print(\'v3 model code imports OK\')"; splat-transform --version', cwd=V3_DIR, env=V3_ENV)

In [ ]:
#@title 1.5 · Download the weights (16.4 GB)
DOWNLOAD_SKY_MODEL = True  #@param {type:"boolean"}
#@markdown From [ritianyu/SplatAnything](https://huggingface.co/ritianyu/SplatAnything): the Gaussian model `SplatAnything.ckpt` (11.3 GB) and the WorldMirror-2 geometry model `worldmirror2.safetensors` (5.1 GB). `DOWNLOAD_SKY_MODEL` adds the 176 MB `skyseg.onnx`, needed only for `SKY_REMOVAL`. The six example scenes ship with the code.
require(V3_PY, '1.4')
missing = [f for f in V3_WEIGHTS + ('skyseg.onnx',) * DOWNLOAD_SKY_MODEL if not (V3_CKPTS / f).is_file()]
if missing:
    sh(f'hf download {V3_HF_REPO} {" ".join(missing)} --local-dir {V3_CKPTS}', env=V3_ENV)
list_files(V3_CKPTS)
for scene in EXAMPLES:
    print(f'example {scene:8s} {len(list_images(V3_DIR / "examples" / scene / "images")):4d} views with cameras')

## 2 · Settings

Cell 2.1 holds every setting the model and its export accept. The defaults are the release's. Run it after any change; every run below uses the current values, and cell 3.6 reruns the last input with them, which makes comparing settings easy. Each output folder records what it used in `config.yaml` and `run.json`.

| Group | Settings | Release default | Where it lives |
| --- | --- | --- | --- |
| Gaussian merging | `VOXEL_SIZE` | 0.002 | `configs/inference.yaml` |
| Scale filter | `FILTER_LARGE_SCALES`, `MAX_SCALE_PERCENTILE`, `MAX_SCALE_ABS` | on, 99.8, off | `configs/inference.yaml` |
| Depth-edge filter | `FILTER_FLYING_POINTS`, `FLYING_POINTS_TOLERANCE`, `FLYING_POINTS_KERNEL` | off, 0.03, 3 | YAML; tolerance and kernel are code defaults of `flying_point_mask` |
| Export | `OPACITY_THRESHOLD`, `EXPORT_SCALE_QUANTILE`, `EXPORT_DEPTH_RGB` | 0.05, off, on | `--opacity_thresh`, `save_gs_ply`, `--no_depth_rgb` |
| Sky | `SKY_REMOVAL`, `SKY_THRESHOLD`, `SKY_MAX_FRACTION` | off, 0.5, 0.95 | `--sky_seg`, `SkySegmenter`, `--sky_max_fraction` |
| Cameras and scale | `KEEP_CAMERA_FRAME`, `SPACING_TARGET` | off, 0.0013227513 | `--no_cam_align`, `GaussianWorldMirror` |
| Memory and speed | `ON_OOM`, `DECODE_CHUNK`, `DEPTH_CHUNK`, `TF32` | fewer views, 8, 8, off | notebook; model chunk sizes; PyTorch |
| Viewer | `EXPORT_HTML`, `VIEWER_MAX_SPLATS` | on, 6M | notebook |

The camera mode (`rgb` or `pose`) and the resolution are chosen per input, in the cells of section 3.

In [ ]:
#@title 2.1 · Model, filtering and export settings (used by every run below; run again after a change)
#@markdown ### Gaussian merging and scale filtering
VOXEL_SIZE = 0.002  #@param {type:"number"}
FILTER_LARGE_SCALES = True  #@param {type:"boolean"}
MAX_SCALE_PERCENTILE = 99.8  #@param {type:"number"}
MAX_SCALE_ABS = 0  #@param {type:"number"}
#@markdown * `VOXEL_SIZE`: the per-pixel Gaussians are merged in voxels of this size, in the normalised scene frame. Smaller keeps more, finer Gaussians and a larger file; larger gives a lighter, coarser scene.
#@markdown * With `FILTER_LARGE_SCALES`, Gaussians larger than the `MAX_SCALE_PERCENTILE` percentile of the scene (`0` = off) or than `MAX_SCALE_ABS` in normalised units (`0` = off) are dropped before merging.
#@markdown ### Depth-edge ("flying points") filter
FILTER_FLYING_POINTS = False  #@param {type:"boolean"}
FLYING_POINTS_TOLERANCE = 0.03  #@param {type:"number"}
FLYING_POINTS_KERNEL = 3  #@param {type:"integer"}
#@markdown * Drops pixels at depth discontinuities, which otherwise leave streaks between foreground and background. A pixel is dropped when the depth spread in its `FLYING_POINTS_KERNEL`×`FLYING_POINTS_KERNEL` window (odd) exceeds `FLYING_POINTS_TOLERANCE` × its depth.
#@markdown ### Export
OPACITY_THRESHOLD = 0.05  #@param {type:"number"}
EXPORT_SCALE_QUANTILE = 0  #@param {type:"number"}
EXPORT_DEPTH_RGB = True  #@param {type:"boolean"}
#@markdown * `OPACITY_THRESHOLD`: Gaussians less opaque than this are left out of `gaussians.ply` (`points.ply` keeps every filtered pixel).
#@markdown * `EXPORT_SCALE_QUANTILE` (for example `0.98`) also leaves out the largest-scale tail of `gaussians.ply`; `0` exports them all.
#@markdown * `EXPORT_DEPTH_RGB` writes each view's processed image and predicted depth (`rgb/`, `depth/`).
#@markdown ### Sky removal
SKY_REMOVAL = False  #@param {type:"boolean"}
SKY_THRESHOLD = 0.5  #@param {type:"number"}
SKY_MAX_FRACTION = 0.95  #@param {type:"number"}
#@markdown * `SKY_REMOVAL` segments the sky in every view with `skyseg.onnx` (cell 1.5), drops sky pixels before merging and keeps them out of the scale estimate. Useful outdoors (street, aerial).
#@markdown * A pixel is sky when its sky probability exceeds `SKY_THRESHOLD`. A view whose sky fraction exceeds `SKY_MAX_FRACTION` is left unfiltered, as a guard against false positives.
#@markdown ### Cameras and scale
KEEP_CAMERA_FRAME = False  #@param {type:"boolean"}
SPACING_TARGET = 0.0013227513  #@param {type:"number"}
#@markdown * In `pose` mode, `KEEP_CAMERA_FRAME` keeps your cameras' world orientation and origin instead of re-anchoring the scene to the first camera; the scale is still normalised.
#@markdown * `SPACING_TARGET` is the median per-pixel spacing (depth ÷ focal length) that Gaussian spacing normalisation scales every scene to. The release value matches what the model was trained with and the voxel size, so change it only to experiment.
#@markdown ### Memory and speed
ON_OOM = 'fewer views'  #@param ["fewer views", "lower resolution", "stop"]
DECODE_CHUNK = 8  #@param {type:"integer"}
DEPTH_CHUNK = 8  #@param {type:"integer"}
TF32 = False  #@param {type:"boolean"}
#@markdown * If a run does not fit in GPU memory, `ON_OOM` decides:
#@markdown   * `fewer views` keeps the largest evenly spaced subset that fits;
#@markdown   * `lower resolution` keeps every view and uses the largest resolution that fits (same aspect ratio);
#@markdown   * `stop` reports the error.
#@markdown * `DECODE_CHUNK` and `DEPTH_CHUNK` are how many views the Gaussian decoder and the depth head process at a time. Lower values use less GPU memory; higher ones can be a little faster. The transformer always attends across all views at once.
#@markdown * `TF32` lets the A100's tensor cores run the FP32 parts (depth, camera and Gaussian heads) faster, at slightly lower precision; PyTorch and the release leave it off.
#@markdown ### Viewer
EXPORT_HTML = True  #@param {type:"boolean"}
VIEWER_MAX_SPLATS = 6000000  #@param {type:"integer"}
#@markdown * `EXPORT_HTML` builds a standalone `scene.html` viewer after each run. Above `VIEWER_MAX_SPLATS` Gaussians, the viewer shows a random subset so the browser stays responsive; `gaussians.ply` always keeps them all.

SETTINGS = dict(
    voxel_size=VOXEL_SIZE, filter_large_scales=FILTER_LARGE_SCALES, max_scale_pct=MAX_SCALE_PERCENTILE,
    max_scale_abs=MAX_SCALE_ABS, filter_flying_points=FILTER_FLYING_POINTS, flying_rtol=FLYING_POINTS_TOLERANCE,
    flying_kernel=FLYING_POINTS_KERNEL, opacity_threshold=OPACITY_THRESHOLD,
    export_scale_quantile=EXPORT_SCALE_QUANTILE, export_depth_rgb=EXPORT_DEPTH_RGB, sky_removal=SKY_REMOVAL,
    sky_threshold=SKY_THRESHOLD, sky_max_fraction=SKY_MAX_FRACTION, keep_camera_frame=KEEP_CAMERA_FRAME,
    spacing_target=SPACING_TARGET, on_oom=OOM_CHOICES[ON_OOM], decode_chunk=DECODE_CHUNK, depth_chunk=DEPTH_CHUNK,
    tf32=TF32, export_html=EXPORT_HTML, viewer_max_splats=VIEWER_MAX_SPLATS)
problems = [message for bad, message in [
    (VOXEL_SIZE <= 0, 'VOXEL_SIZE must be positive'),
    (not 0 <= MAX_SCALE_PERCENTILE <= 100, 'MAX_SCALE_PERCENTILE must be from 0 (off) to 100'),
    (MAX_SCALE_ABS < 0, 'MAX_SCALE_ABS must be 0 (off) or positive'),
    (FLYING_POINTS_TOLERANCE <= 0, 'FLYING_POINTS_TOLERANCE must be positive'),
    (FLYING_POINTS_KERNEL < 1 or FLYING_POINTS_KERNEL % 2 == 0, 'FLYING_POINTS_KERNEL must be an odd number'),
    (not 0 <= OPACITY_THRESHOLD <= 1, 'OPACITY_THRESHOLD must be from 0 to 1'),
    (not 0 <= EXPORT_SCALE_QUANTILE < 1, 'EXPORT_SCALE_QUANTILE must be 0 (off) or a fraction below 1'),
    (not 0 <= SKY_THRESHOLD <= 1, 'SKY_THRESHOLD must be from 0 to 1'),
    (not 0 <= SKY_MAX_FRACTION <= 1, 'SKY_MAX_FRACTION must be from 0 to 1'),
    (SPACING_TARGET <= 0, 'SPACING_TARGET must be positive'),
    (DECODE_CHUNK < 1 or DEPTH_CHUNK < 1, 'DECODE_CHUNK and DEPTH_CHUNK must be at least 1'),
    (VIEWER_MAX_SPLATS < 1, 'VIEWER_MAX_SPLATS must be at least 1'),
] if bad]
if problems:
    raise ValueError('; '.join(problems))
changed = {k: v for k, v in SETTINGS.items() if v != DEFAULT_SETTINGS[k]}
print('Settings saved:', 'the release defaults.' if not changed else
      'changed from the release: ' + ', '.join(f'{k}={v}' for k, v in changed.items()))

## 3 · Reconstruct

Each run makes a new folder under `/content/outputs/runs/`, and the cell then shows:
* the run summary, including every out-of-memory attempt;
* point previews from the first camera and two orbiting views;
* processed images above their predicted depth;
* an interactive 3D point plot;
* the HTML viewer, which renders the real Gaussians.

Camera modes:
* `rgb`: images only; the cameras are predicted.
* `pose`: calibrated cameras condition the reconstruction. They come from a `prior_camera.json`, or from a COLMAP model that the notebook converts for you.

In [ ]:
#@title 3.1 · Reconstruct an example scene
EXAMPLE = 'object'  #@param ["object", "indoor", "street", "aerial", "rig", "pano"]
MODE = 'pose'  #@param ["pose", "rgb"]
NUM_VIEWS = 0  #@param {type:"integer"}
RESOLUTION = 'release setting (756×756; rig 504×504)'  #@param ["release setting (756\u00d7756; rig 504\u00d7504)", "match aspect, 518\u00b2 px", "match aspect, 756\u00b2 px", "match aspect, 1022\u00b2 px", "match aspect, 1260\u00b2 px", "match aspect, 1512\u00b2 px", "square 518\u00d7518", "square 756\u00d7756 (release default, centre crop)", "square 1022\u00d71022", "custom WIDTH \u00d7 HEIGHT"]
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown * `object` (23 views), `indoor` (16), `street` (100), `aerial` (100), `rig` (150 views from the surround cameras of a driving rig), `pano` (24 views cut from one 360° panorama). Every example ships calibrated cameras.
#@markdown * `pose` uses the example's cameras; `rgb` ignores them and predicts the cameras from the images.
#@markdown * `NUM_VIEWS = 0` uses every view; a number picks that many, evenly spaced.
#@markdown * `RESOLUTION`: `match aspect` keeps the input's aspect ratio at about N×N pixels (nothing is cropped); `square` sizes are scaled to cover and centre-cropped, like the release; `custom` uses `WIDTH` × `HEIGHT` (rounded to multiples of 14). There is no upper limit; more pixels give more detail and cost memory like more views do.
#@markdown * Model, filtering, export and memory settings come from cell 2.1.

scene_dir = V3_DIR / 'examples' / EXAMPLE
LAST_RESULT = reconstruct(list_images(scene_dir / 'images'), EXAMPLE, mode=MODE,
                          cameras=scene_dir / 'prior_camera.json' if MODE == 'pose' else None, num_views=NUM_VIEWS,
                          resolution=RESOLUTION, width=WIDTH, height=HEIGHT,
                          release=(504, 504) if EXAMPLE == 'rig' else (756, 756))

In [ ]:
#@title 3.2 · Look at the latest result
show_result(LAST_RESULT, html=SETTINGS['export_html'])

In [ ]:
#@title 3.3 · Reconstruct your own photos
IMAGES_DIR = ''  #@param {type:"string"}
MODE = 'rgb'  #@param ["rgb", "pose"]
CAMERAS = ''  #@param {type:"string"}
NUM_VIEWS = 0  #@param {type:"integer"}
RESOLUTION = 'match aspect, 756² px'  #@param ["match aspect, 518\u00b2 px", "match aspect, 756\u00b2 px", "match aspect, 1022\u00b2 px", "match aspect, 1260\u00b2 px", "match aspect, 1512\u00b2 px", "square 518\u00d7518", "square 756\u00d7756 (release default, centre crop)", "square 1022\u00d71022", "custom WIDTH \u00d7 HEIGHT"]
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown * Leave `IMAGES_DIR` empty to upload photos or a `.zip` of photos, or give a folder or `.zip` path (for Google Drive, mount Drive first with the folder icon on the left). Use overlapping JPG/PNG/WebP/BMP/TIFF photos of one static scene — there is no upper limit. They are read in filename order, and file names (without extension) must be unique. Convert HEIC photos to JPG first. For hundreds of photos, a `.zip` on Drive is much faster than a browser upload.
#@markdown * `rgb` predicts the cameras. `pose` needs `CAMERAS`: either a `prior_camera.json` (OpenCV camera-to-world poses and pixel intrinsics of the original images, keyed by file name without extension; see `InfiniSplat_v3/docs/prepare_data.md`), or the folder of an undistorted COLMAP sparse model (`PINHOLE` or `SIMPLE_PINHOLE`, e.g. `.../sparse/0`), which is converted for you. A multi-camera rig is the same: all its images and their cameras.
#@markdown * `NUM_VIEWS = 0` uses every photo; a number picks that many, evenly spaced.
#@markdown * `RESOLUTION`: `match aspect` keeps the input's aspect ratio at about N×N pixels (nothing is cropped); `square` sizes are scaled to cover and centre-cropped, like the release; `custom` uses `WIDTH` × `HEIGHT` (rounded to multiples of 14). There is no upper limit; more pixels give more detail and cost memory like more views do.
#@markdown * Model, filtering, export and memory settings come from cell 2.1.

if MODE == 'pose' and not CAMERAS:
    raise ValueError('pose mode needs CAMERAS (a prior_camera.json or a COLMAP model folder); or use MODE = rgb.')
source = image_source(IMAGES_DIR, 'photos')
images = list_images(source)
if not images:
    raise ValueError(f'No JPG/PNG/WebP/BMP/TIFF images found in {source}.')
name = Path(IMAGES_DIR).stem if IMAGES_DIR else 'upload'
LAST_RESULT = reconstruct(images, name, mode=MODE, cameras=CAMERAS or None, num_views=NUM_VIEWS,
                          resolution=RESOLUTION, width=WIDTH, height=HEIGHT)
show_result(LAST_RESULT, html=SETTINGS['export_html'])

In [ ]:
#@title 3.4 · Reconstruct from a video
VIDEO_PATH = ''  #@param {type:"string"}
NUM_FRAMES = 0  #@param {type:"integer"}
START_SECONDS = 0  #@param {type:"number"}
END_SECONDS = 0  #@param {type:"number"}
RESOLUTION = 'match aspect, 756² px'  #@param ["match aspect, 518\u00b2 px", "match aspect, 756\u00b2 px", "match aspect, 1022\u00b2 px", "match aspect, 1260\u00b2 px", "match aspect, 1512\u00b2 px", "square 518\u00d7518", "square 756\u00d7756 (release default, centre crop)", "square 1022\u00d71022", "custom WIDTH \u00d7 HEIGHT"]
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown * Leave `VIDEO_PATH` empty to upload a video (MP4, MOV, M4V, WebM, MKV, AVI), or give a path, for example on Google Drive (faster for large files). Move slowly around a static scene; motion blur and moving objects hurt the result.
#@markdown * The clip (optionally trimmed to `START_SECONDS`–`END_SECONDS`; `0` = whole clip) is split into `NUM_FRAMES` equal time slices, and the sharpest frame of each slice is kept; every kept frame is used. Any `NUM_FRAMES` is accepted (a clip cannot give more frames than it contains). `NUM_FRAMES = 0` takes 100, like the largest bundled photo examples.
#@markdown * Video frames have no calibrated cameras, so v3 predicts them (`rgb` mode).
#@markdown * `RESOLUTION`: `match aspect` keeps the input's aspect ratio at about N×N pixels (nothing is cropped); `square` sizes are scaled to cover and centre-cropped, like the release; `custom` uses `WIDTH` × `HEIGHT` (rounded to multiples of 14). There is no upper limit; more pixels give more detail and cost memory like more views do.
#@markdown * Model, filtering, export and memory settings come from cell 2.1.

if VIDEO_PATH:
    video = Path(VIDEO_PATH)
else:
    video = next((p for p in upload_files('video').iterdir() if p.suffix.lower() in VIDEO_EXTS), None)
    if video is None:
        raise ValueError(f'Upload one video ({", ".join(sorted(VIDEO_EXTS))}).')
frames = extract_frames(video, NUM_FRAMES or 100, start=START_SECONDS, end=END_SECONDS)
LAST_RESULT = reconstruct(list_images(frames), video.stem, mode='rgb', resolution=RESOLUTION, width=WIDTH,
                          height=HEIGHT)
show_result(LAST_RESULT, html=SETTINGS['export_html'])

In [ ]:
#@title 3.5 · Reconstruct a 360° panorama
PANORAMA = 'bundled example (examples/pano/pano.jpg)'  #@param ["bundled example (examples/pano/pano.jpg)", "my panorama"]
PANORAMA_PATH = ''  #@param {type:"string"}
VIEW_SIZE = 1022  #@param {type:"integer"}
VIEW_FOV = 60  #@param {type:"number"}
MODE = 'pose'  #@param ["pose", "rgb"]
RESOLUTION = 'match aspect, 756² px'  #@param ["match aspect, 518\u00b2 px", "match aspect, 756\u00b2 px", "match aspect, 1022\u00b2 px", "match aspect, 1260\u00b2 px", "match aspect, 1512\u00b2 px", "square 518\u00d7518", "square 756\u00d7756 (release default, centre crop)", "square 1022\u00d71022", "custom WIDTH \u00d7 HEIGHT"]
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown * The repository's `scripts/convert_panorama.py` cuts a full 360°×180° equirectangular image (exactly 2:1, no EXIF rotation) into 24 calibrated perspective views: 8 directions at pitch 0°, −45° and +45°, each `VIEW_SIZE` px square with a `VIEW_FOV`° field of view. `pose` uses their exact cameras (the release's way); `rgb` predicts them.
#@markdown * With `my panorama`, leave `PANORAMA_PATH` empty to upload the image, or give a path.
#@markdown * `RESOLUTION`: `match aspect` keeps the input's aspect ratio at about N×N pixels (nothing is cropped); `square` sizes are scaled to cover and centre-cropped, like the release; `custom` uses `WIDTH` × `HEIGHT` (rounded to multiples of 14). There is no upper limit; more pixels give more detail and cost memory like more views do.
#@markdown * Model, filtering, export and memory settings come from cell 2.1.

if PANORAMA.startswith('bundled'):
    panorama = V3_DIR / 'examples' / 'pano' / 'pano.jpg'
elif PANORAMA_PATH:
    panorama = Path(PANORAMA_PATH)
else:
    panorama = next((p for p in upload_files('panorama').iterdir() if p.suffix.lower() in IMAGE_EXTS), None)
    if panorama is None:
        raise ValueError('Upload one equirectangular panorama image.')
views = OUTPUTS / 'panorama_views' / f'{panorama.stem}-{time.strftime("%Y%m%d-%H%M%S")}'
sh(shlex.join(map(str, [V3_PY, V3_DIR / 'scripts' / 'convert_panorama.py', '--input', panorama.resolve(),
                        '--output', views, '--size', VIEW_SIZE, '--fov', VIEW_FOV])), env=V3_ENV)
LAST_RESULT = reconstruct(list_images(views / 'images'), f'{panorama.stem}-panorama', mode=MODE,
                          cameras=views / 'prior_camera.json' if MODE == 'pose' else None, resolution=RESOLUTION,
                          width=WIDTH, height=HEIGHT)
show_result(LAST_RESULT, html=SETTINGS['export_html'])

In [ ]:
#@title 3.6 · Run the last input again with the current settings
MODE = 'same as before'  #@param ["same as before", "rgb", "pose"]
NUM_VIEWS = -1  #@param {type:"integer"}
RESOLUTION = 'same as before'  #@param ["same as before", "match aspect, 518\u00b2 px", "match aspect, 756\u00b2 px", "match aspect, 1022\u00b2 px", "match aspect, 1260\u00b2 px", "match aspect, 1512\u00b2 px", "square 518\u00d7518", "square 756\u00d7756 (release default, centre crop)", "square 1022\u00d71022", "custom WIDTH \u00d7 HEIGHT"]
WIDTH = 0  #@param {type:"integer"}
HEIGHT = 0  #@param {type:"integer"}
#@markdown * Change settings in cell 2.1 and run it, then run this cell to compare: it reconstructs the same input (example, photos, video frames or panorama views) into a new folder with the current settings. The previous result stays where it is.
#@markdown * `NUM_VIEWS = -1` keeps the previous choice; `0` uses every view.
#@markdown * `RESOLUTION`: `match aspect` keeps the input's aspect ratio at about N×N pixels (nothing is cropped); `square` sizes are scaled to cover and centre-cropped, like the release; `custom` uses `WIDTH` × `HEIGHT` (rounded to multiples of 14). There is no upper limit; more pixels give more detail and cost memory like more views do.

if not LAST_INPUT:
    raise ValueError('Nothing to rerun yet: run one of cells 3.1, 3.3, 3.4 or 3.5 first.')
inputs = dict(LAST_INPUT)
if MODE != 'same as before':
    if MODE == 'pose' and not inputs['cameras']:
        raise ValueError('The last input has no cameras, so it can only run in rgb mode.')
    inputs['mode'] = MODE
if NUM_VIEWS >= 0:
    inputs['num_views'] = NUM_VIEWS
if RESOLUTION != 'same as before':
    inputs.update(resolution=RESOLUTION, width=WIDTH, height=HEIGHT)
LAST_RESULT = reconstruct(**inputs)
show_result(LAST_RESULT, html=SETTINGS['export_html'])

## 4 · Keep your results

Everything is written under `/content/outputs` (`runs/` holds one folder per run), which disappears when the Colab runtime is recycled. Each run folder contains:

| File | Contents |
| --- | --- |
| `gaussians.ply` | The Gaussian scene, in standard 3DGS PLY format (SH degree 0); opens in most splat viewers |
| `points.ply` | Every filtered pixel as a coloured point, before voxel merging |
| `cameras.json` | Camera-to-world poses (OpenCV) and intrinsics of the processed images, in the cloud's frame |
| `rgb/`, `depth/` | Each view as processed by the model, and its predicted depth (`.npy` in the geometry model's scale, `.png` preview) |
| `scene.html` | Standalone viewer (open it in a browser) |
| `config.yaml`, `run.json`, `notebook_run.json` | The settings used, the inputs, and every out-of-memory attempt |

In [ ]:
#@title 4.1 · Download the latest result
WHAT = 'everything (zip)'  #@param ["everything (zip)", "gaussians.ply", "points.ply", "viewer (html)"]
if WHAT.startswith('everything'):
    download(LAST_RESULT)
elif WHAT.endswith('.ply'):
    download(LAST_RESULT / WHAT)
else:
    preview_html = LAST_RESULT / 'viewer' / 'scene_preview.html'
    download(preview_html if preview_html.is_file() else LAST_RESULT / 'scene.html')

In [ ]:
#@title 4.2 · Copy all results to Google Drive
DRIVE_FOLDER = 'InfiniSplat_v3_outputs'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
target = Path('/content/drive/MyDrive') / DRIVE_FOLDER
# Scene folders only hold links to the inputs; copy the results, settings and extracted video/panorama frames.
shutil.copytree(OUTPUTS, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns('uploads', 'scenes'))
print('Copied to', target)

## Troubleshooting

| Symptom | What to do |
| --- | --- |
| `CUDA OK` is missing, or a run prints *barely used the GPU* | Choose *Runtime → Change runtime type → A100 GPU*, then *Runtime → Disconnect and delete runtime*, and run again from 1.1. On an older driver, rerun 1.4 with `TORCH_CUDA = cu126`. |
| `CUDA out of memory` | The run retries by itself (`ON_OOM` in 2.1). With `stop`, lower `NUM_VIEWS` or the resolution, or lower `DECODE_CHUNK`/`DEPTH_CHUNK`. |
| A run used fewer views than you gave it | All of them did not fit in GPU memory, so `fewer views` kept the largest evenly spaced subset that did; the run summary lists every attempt. To keep every view, use `ON_OOM = lower resolution`, or a smaller `RESOLUTION`. |
| Exit code `-9` / `137`, or the runtime restarts | Out of system RAM while loading the weights. Use an A100 or High-RAM runtime. |
| `NameError` after a runtime restart | Rerun cells 1.1–1.3 and 2.1. The environment and weights stay on disk until the runtime is recycled. |
| An install or download cell failed halfway | Run it again; the install starts from scratch until it succeeds, and downloads resume. |
| `pose` mode reports a missing camera entry or a principal point outside the image | Every image needs a camera keyed by its file name without extension, with intrinsics in that image's own pixels. Use `rgb` mode, or check the cameras with `scripts/convert_cameras.py check` (see the data preparation guide). |
| *All Gaussians were filtered* | Usually `SKY_REMOVAL` on a scene that is mostly sky, or very aggressive scale or depth-edge filters; relax them in 2.1. |
| A photo comes out sideways | v3 ignores EXIF rotation; the notebook stores rotated photos upright in `rgb` mode. In `pose` mode, rotate the images yourself to match your cameras. |
| The viewer stays blank | Large scenes take a while to load; use the *Open the viewer in a new browser tab* link, or download `scene.html` and open it locally. The point previews never depend on the viewer. |
| Poor reconstruction | Check overlap, motion blur and exposure. With calibrated cameras, try `pose` mode. Outdoors, try `SKY_REMOVAL`; for streaks at object edges, try `FILTER_FLYING_POINTS`. More pixels (`RESOLUTION`) give more detail. |

More details: [v3 README](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v3/README.md) · [inference reference](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v3/docs/inference.md) · [data preparation](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v3/docs/prepare_data.md)